# 07b — Model comparison with repeated patient-grouped CV (full MIMIC-IV)

**Question:** which of Logistic Regression / Random Forest / XGBoost is actually better on the full
cohort, and how much of the difference is real vs. a lucky split?

**Method:** 5×5 repeated `StratifiedGroupKFold` (group = `subject_id`) on the training pool
(train + validation). The frozen test set is never touched here.

Knowledge points: 14 (imbalance), 15 (model comparison), 16 (cross-validation),
17 (bias–variance), 20 (bootstrap / CI).

In [ ]:
import sys
from pathlib import Path

PROJECT_DIR = Path.cwd().parent
sys.path.insert(0, str(PROJECT_DIR))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.config import RANDOM_STATE, TARGET_COLUMN, TABLES_DIR, FIGURES_DIR
from src.features.preprocessing import infer_feature_types, build_preprocessor
from src.models.logistic import build_logistic_model
from src.models.random_forest import build_random_forest
from src.models.xgboost_model import build_xgboost, calculate_scale_pos_weight
from src.evaluation.resampling import repeated_grouped_cv, summarize_cv_results

# QUICK=True -> 2 repeats x 3 folds (6 fits per model, ~5 min). Set False for the real 5x5 run.
QUICK = True
N_REPEATS, N_SPLITS = (2, 3) if QUICK else (5, 5)

## 1. Load the full cohort and freeze a patient-level test set

If you have already run notebook 03 on the full data, replace this cell with `load_train_data()` +
`load_validation_data()` and concatenate them. Until then, we hold out 20% of *patients* here with a
fixed seed and save their IDs so later notebooks freeze the same test set.

In [ ]:
df = pd.read_csv(PROJECT_DIR / "data/raw/full/modeling_cohort_full.csv")
print(df.shape, "mortality:", round(df[TARGET_COLUMN].mean(), 4))

rng = np.random.default_rng(RANDOM_STATE)
subjects = df["subject_id"].unique()
test_subjects = rng.choice(subjects, size=int(0.2 * len(subjects)), replace=False)

test_ids_path = PROJECT_DIR / "data/processed/full/test_subject_ids.csv"
test_ids_path.parent.mkdir(parents=True, exist_ok=True)
pd.Series(test_subjects, name="subject_id").to_csv(test_ids_path, index=False)

pool = df[~df["subject_id"].isin(test_subjects)].reset_index(drop=True)
print("training pool:", pool.shape, "positives:", int(pool[TARGET_COLUMN].sum()))

## 2. Features and preprocessor (same choices as notebook 05)

In [ ]:
ID_AND_LEAKAGE = ["subject_id", "hadm_id", "stay_id", "intime", "outtime", "prediction_time", TARGET_COLUMN]
feature_columns = [c for c in pool.columns if c not in ID_AND_LEAKAGE]

numeric_features, categorical_features = infer_feature_types(pool, feature_columns)
print(len(numeric_features), "numeric |", len(categorical_features), "categorical")

preprocessor = build_preprocessor(numeric_features, categorical_features)

X = pool[feature_columns]
y = pool[TARGET_COLUMN].to_numpy()
groups = pool["subject_id"].to_numpy()

## 3. Three candidate pipelines

Same hyper-parameters as notebooks 06–07 so the comparison is with *your* models, not new ones.
XGBoost's `scale_pos_weight` is computed on the pool (it is a training-set quantity).

In [ ]:
models = {
    "logistic": build_logistic_model(preprocessor),
    "random_forest": build_random_forest(preprocessor),
    "xgboost": build_xgboost(preprocessor, scale_pos_weight=calculate_scale_pos_weight(y)),
}

## 4. Repeated grouped CV

This is the slow cell. On 68k rows: LR ≈ seconds per fit, RF/XGB ≈ 30–90 s per fit.

In [ ]:
import time

all_results = []
for name, pipeline in models.items():
    t0 = time.time()
    res = repeated_grouped_cv(
        pipeline, X, y, groups,
        n_splits=N_SPLITS, n_repeats=N_REPEATS, model_name=name,
    )
    all_results.append(res)
    print(f"{name:14s} {len(res)} fits  AUROC {res['auroc'].mean():.3f} ± {res['auroc'].std():.3f}  ({time.time()-t0:.0f}s)")

cv_results = pd.concat(all_results, ignore_index=True)
cv_results.to_csv(TABLES_DIR / "full_cv_fold_results.csv", index=False)

## 5. Summary table — mean, SD and 2.5/97.5 percentiles across folds

In [ ]:
summary = summarize_cv_results(cv_results)
summary.to_csv(TABLES_DIR / "full_cv_model_comparison.csv", index=False)
summary.round(4)

## 6. Boxplot — is the gap between models bigger than the fold-to-fold spread?

Read it like this: if two boxes overlap heavily, the models are not distinguishable on this data.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, metric in zip(axes, ["auroc", "auprc", "brier"]):
    data = [cv_results.loc[cv_results["model"] == m, metric] for m in models]
    ax.boxplot(data, tick_labels=list(models))
    ax.set_title(metric.upper())
    ax.grid(alpha=0.3)
fig.suptitle(f"{N_REPEATS}x{N_SPLITS} repeated patient-grouped CV — full MIMIC-IV cohort")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "full_cv_model_comparison.png", dpi=150)
plt.show()

## 7. Paired comparison — same folds, so differences are less noisy

Because every model saw the *same* 25 splits, we can look at per-fold differences directly.
A difference whose 95% range excludes 0 is a real difference on this data.

In [ ]:
wide = cv_results.pivot_table(index=["repeat", "fold"], columns="model", values="auroc")
diffs = pd.DataFrame({
    "rf - logistic":  wide["random_forest"] - wide["logistic"],
    "xgb - logistic": wide["xgboost"] - wide["logistic"],
    "xgb - rf":       wide["xgboost"] - wide["random_forest"],
})
diffs.describe(percentiles=[0.025, 0.5, 0.975]).T[["mean", "2.5%", "50%", "97.5%"]].round(4)

## 8. Write-up (fill in after the run)

- Best model by mean AUROC: _0.891__ ; by AUPRC: _0.599__
- Is the best model's advantage larger than the fold-to-fold SD?
  Yes — XGBoost beats logistic by 0.022 AUROC, about 5× the fold SD (~0.004); the 2.5–97.5 percentile ranges of the three models do not overlap.
 ___
- Compare to the demo result (RF 0.94 / LR 0.88 on 20 validation rows): 
  the ranking reversed. On the full cohort RF (0.855) is the weakest model, below logistic (0.868). The demo "RF wins" was split noise on 2 positive cases, not a real finding.___
- What changes in your model-selection decision, if anything:
  witch `selected_model` from Random Forest to XGBoost. Also revisit RF hyper-parameters (max_depth=8, min_samples_leaf=5 were chosen for 89 training rows and now under-fit) before the final comparison in notebook 07. ___

I compared models with repeated patient-grouped cross-validation rather than a
single split, because with a single split I couldn't tell whether a 0.02 AUROC gap was signal or
noise. Grouping by patient prevents leakage across folds; repeating the split lets me report a
spread, and paired per-fold differences let me say whether one model is *reliably* better.
